# PartCrafter on Google Colab — image → parts-separated GLB

**Before running:** menu **Runtime → Change runtime type → Hardware accelerator = T4 GPU**, then **Runtime → Run all**.

Steps: (1) check GPU → (2) install PartCrafter (~5–15 min) → (3) upload your `mecha_src.png` → (4) generate parts → (5) download the GLB.
Then hand the downloaded `.glb` back to the assistant for rigging (①〜④ + joint cores).

In [ ]:
# 1) Confirm a GPU is attached (must show a Tesla T4 or similar)
!nvidia-smi

In [ ]:
# 2) Clone + install PartCrafter (weights auto-download from Hugging Face on first run)
%cd /content
![ -d PartCrafter ] || git clone https://github.com/wgsxm/PartCrafter.git
%cd /content/PartCrafter
# Pin torch to PartCrafter's spec (2.5.1+cu124) FIRST — Colab now ships torch 2.11+cu128,
# incompatible with the prebuilt torch_cluster wheel.
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
# official setup
!bash settings/setup.sh
# Colab ships transformers 5.x + huggingface_hub 1.x + tokenizers 0.22 which break
# diffusers 0.38 (AutoImageProcessor import). Pin a compatible 4.x set AFTER setup.
!pip install -q "transformers==4.49.0" "huggingface_hub==0.27.1" "tokenizers==0.21.0" "diffusers==0.38.0"
# The prebuilt torch_cluster wheel ABI-mismatches the installed torch (undefined symbol).
# Rebuild torch_cluster FROM SOURCE against the installed torch (guaranteed ABI match). ~3-8 min on T4.
import os, torch
os.environ['CUDA_HOME'] = '/usr/local/cuda'; os.environ['FORCE_CUDA'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'  # T4 = sm_75
get_ipython().system('pip uninstall -y -q torch-cluster')
get_ipython().system('pip install -q --no-cache-dir --no-build-isolation torch-cluster')
print('\n>>> install cell finished <<<')

In [ ]:
# 2b) Fast import check — imports the FULL pipeline (incl. torch_cluster) so any version/ABI
# mismatch shows up in seconds, before the slow weight download. Must print 'IMPORTS OK'.
!cd /content/PartCrafter && PYTHONPATH=/content/PartCrafter python -c "import torch, transformers, diffusers; from torch_cluster import fps; from src.pipelines.pipeline_partcrafter import PartCrafterPipeline; print('IMPORTS OK | torch', torch.__version__, '| transformers', transformers.__version__, '| diffusers', diffusers.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# 3) Upload your input image (the SD-generated mecha_src.png)
from google.colab import files
print('Choose mecha_src.png ...')
up = files.upload()
IMG = list(up.keys())[0]
print('uploaded:', IMG)

In [ ]:
# 4) Generate parts.  num_parts: try 8 for a humanoid mecha (head/chest/hips/2 arms/2 legs/...). Max 16.
NUM_PARTS = 8  #@param {type:"integer"}
%cd /content/PartCrafter
# PYTHONPATH=repo root so 'import src' resolves (running scripts/ alone is not enough)
!PYTHONPATH=/content/PartCrafter python scripts/inference_partcrafter.py --image_path "/content/PartCrafter/{IMG}" --num_parts {NUM_PARTS} --tag mecha --render
# if it OOMs on T4, lower NUM_PARTS or add: --num_tokens 768

In [ ]:
# 5) Find the generated GLB(s) and download.  Hand the .glb back to the assistant.
import glob, os
glbs = sorted(glob.glob('/content/PartCrafter/results/**/*.glb', recursive=True), key=os.path.getmtime)
print('found GLB files:')
for g in glbs: print('  ', g, os.path.getsize(g), 'bytes')
from google.colab import files
if glbs:
    files.download(glbs[-1])
else:
    print('No GLB found — check the cell 4 output for errors (e.g. OOM); lower num_parts and re-run.')